# Bhojpuri GPT-2 Large Training (Colab T4)

This notebook is set up to fine-tune `gpt2-large` on your Bhojpuri dataset in Google Colab.

Recommended for T4: use `LoRA` mode, not full fine-tuning.

Before running the upload cell, keep these two files ready from your local machine:
- `conditioned_train.jsonl`
- `conditioned_val.jsonl`


In [ ]:
!pip install -q -U transformers datasets accelerate peft sentencepiece


In [ ]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'Please enable GPU in Colab: Runtime > Change runtime type > GPU'


In [ ]:
from pathlib import Path

BASE_DIR = Path('/content/bhojpuri_gpt2')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs' / 'gpt2-large-bhojpuri'

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR =', BASE_DIR)
print('DATA_DIR =', DATA_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)


## Optional: Save Everything To Google Drive

Set `USE_DRIVE = True` in the next cell if you want checkpoints to persist after the Colab session ends.


In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    from pathlib import Path
    BASE_DIR = Path('/content/drive/MyDrive/bhojpuri_gpt2')
    DATA_DIR = BASE_DIR / 'data'
    OUTPUT_DIR = BASE_DIR / 'outputs' / 'gpt2-large-bhojpuri'
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print('Using Google Drive paths')
    print('DATA_DIR =', DATA_DIR)
    print('OUTPUT_DIR =', OUTPUT_DIR)
else:
    print('Using local /content paths')


## Upload The Prepared Dataset Files

Upload these two files from your machine:
- `conditioned_train.jsonl`
- `conditioned_val.jsonl`


In [ ]:
from google.colab import files
uploaded = files.upload()

required = {'conditioned_train.jsonl', 'conditioned_val.jsonl'}
for name, content in uploaded.items():
    (DATA_DIR / name).write_bytes(content)

missing = [name for name in required if not (DATA_DIR / name).exists()]
assert not missing, f'Missing required files: {missing}'

print('Files in DATA_DIR:')
for path in sorted(DATA_DIR.iterdir()):
    print(path.name, path.stat().st_size)


## Write The Training Script

This cell creates `train_gpt2_large_hf.py` inside the Colab session so the notebook stays self-contained.


In [ ]:
%%writefile train_gpt2_large_hf.py
﻿#!/usr/bin/env python
"""Train GPT-2 Large on the prepared Bhojpuri dataset with Hugging Face.

Defaults are tuned for Google Colab T4 by using LoRA, mixed precision,
gradient checkpointing, and small per-device batch sizes.
"""

from __future__ import annotations

import argparse
import inspect
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)


def parse_args(argv: list[str]) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description='Fine-tune GPT-2 Large on Bhojpuri lyrics.')
    parser.add_argument('--model-name', default='gpt2-large', help='Base model name or local path.')
    parser.add_argument(
        '--train-file',
        type=Path,
        default=Path('results') / 'gpt2_bhojpuri' / 'conditioned_train.jsonl',
        help='Training JSONL with a text field.',
    )
    parser.add_argument(
        '--validation-file',
        type=Path,
        default=Path('results') / 'gpt2_bhojpuri' / 'conditioned_val.jsonl',
        help='Validation JSONL with a text field.',
    )
    parser.add_argument(
        '--output-dir',
        type=Path,
        default=Path('outputs') / 'gpt2-large-bhojpuri',
        help='Directory for checkpoints and the final model.',
    )
    parser.add_argument(
        '--train-mode',
        choices=['lora', 'full'],
        default='lora',
        help='Use LoRA for T4-friendly training or full to update all weights.',
    )
    parser.add_argument('--max-length', type=int, default=256, help='Maximum tokenized sequence length.')
    parser.add_argument('--num-train-epochs', type=float, default=5.0, help='Number of training epochs.')
    parser.add_argument('--learning-rate', type=float, default=None, help='Override learning rate.')
    parser.add_argument('--weight-decay', type=float, default=0.01, help='Weight decay.')
    parser.add_argument('--warmup-ratio', type=float, default=0.05, help='Warmup ratio.')
    parser.add_argument('--lr-scheduler-type', default='cosine', help='Learning rate scheduler.')
    parser.add_argument('--per-device-train-batch-size', type=int, default=1, help='Train batch size per device.')
    parser.add_argument('--per-device-eval-batch-size', type=int, default=1, help='Eval batch size per device.')
    parser.add_argument('--gradient-accumulation-steps', type=int, default=16, help='Gradient accumulation steps.')
    parser.add_argument('--logging-steps', type=int, default=10, help='Logging interval in optimizer steps.')
    parser.add_argument('--eval-steps', type=int, default=50, help='Evaluation interval in optimizer steps.')
    parser.add_argument('--save-steps', type=int, default=50, help='Checkpoint save interval in optimizer steps.')
    parser.add_argument('--save-total-limit', type=int, default=2, help='Number of checkpoints to keep.')
    parser.add_argument('--seed', type=int, default=42, help='Random seed.')
    parser.add_argument('--report-to', default='none', help='Trainer report target, e.g. none or wandb.')
    parser.add_argument('--resume-from-checkpoint', default=None, help='Resume from a checkpoint path if needed.')
    parser.add_argument('--disable-gradient-checkpointing', action='store_true', help='Turn off gradient checkpointing.')
    parser.add_argument('--lora-r', type=int, default=8, help='LoRA rank.')
    parser.add_argument('--lora-alpha', type=int, default=16, help='LoRA alpha.')
    parser.add_argument('--lora-dropout', type=float, default=0.05, help='LoRA dropout.')
    return parser.parse_args(argv)



def choose_learning_rate(args: argparse.Namespace) -> float:
    if args.learning_rate is not None:
        return args.learning_rate
    return 2e-4 if args.train_mode == 'lora' else 5e-5



def ensure_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f'{label} not found: {path}')



def count_parameters(model: torch.nn.Module) -> dict[str, int]:
    total = sum(param.numel() for param in model.parameters())
    trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
    return {'total': total, 'trainable': trainable}



def load_text_datasets(train_file: Path, validation_file: Path):
    data_files: dict[str, str] = {'train': str(train_file)}
    if validation_file.exists():
        data_files['validation'] = str(validation_file)
    return load_dataset('json', data_files=data_files)



def build_model_and_tokenizer(args: argparse.Namespace):
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(args.model_name)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False

    if args.train_mode == 'lora':
        from peft import LoraConfig, TaskType, get_peft_model

        if hasattr(model, 'enable_input_require_grads'):
            model.enable_input_require_grads()

        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=args.lora_r,
            lora_alpha=args.lora_alpha,
            lora_dropout=args.lora_dropout,
            target_modules=['c_attn', 'c_proj'],
            fan_in_fan_out=True,
            bias='none',
        )
        model = get_peft_model(model, lora_config)

    return model, tokenizer



def tokenize_datasets(raw_datasets, tokenizer, max_length: int):
    train_columns = raw_datasets['train'].column_names

    def tokenize_batch(batch: dict[str, list[Any]]) -> dict[str, Any]:
        return tokenizer(
            batch['text'],
            truncation=True,
            max_length=max_length,
            padding=False,
        )

    tokenized = raw_datasets.map(
        tokenize_batch,
        batched=True,
        remove_columns=train_columns,
        desc='Tokenizing dataset',
    )
    return tokenized



def build_training_args(args: argparse.Namespace, has_validation: bool) -> TrainingArguments:
    training_kwargs: dict[str, Any] = {
        'output_dir': str(args.output_dir),
        'num_train_epochs': args.num_train_epochs,
        'learning_rate': choose_learning_rate(args),
        'weight_decay': args.weight_decay,
        'warmup_ratio': args.warmup_ratio,
        'lr_scheduler_type': args.lr_scheduler_type,
        'per_device_train_batch_size': args.per_device_train_batch_size,
        'per_device_eval_batch_size': args.per_device_eval_batch_size,
        'gradient_accumulation_steps': args.gradient_accumulation_steps,
        'logging_steps': args.logging_steps,
        'save_steps': args.save_steps,
        'save_total_limit': args.save_total_limit,
        'seed': args.seed,
        'report_to': args.report_to,
        'optim': 'adamw_torch',
        'fp16': torch.cuda.is_available(),
        'bf16': False,
        'gradient_checkpointing': not args.disable_gradient_checkpointing,
        'dataloader_pin_memory': torch.cuda.is_available(),
        'group_by_length': True,
        'logging_first_step': True,
        'remove_unused_columns': False,
    }

    training_signature = inspect.signature(TrainingArguments.__init__).parameters
    eval_strategy_key = 'eval_strategy' if 'eval_strategy' in training_signature else 'evaluation_strategy'

    if has_validation:
        training_kwargs.update(
            {
                eval_strategy_key: 'steps',
                'eval_steps': args.eval_steps,
                'save_strategy': 'steps',
                'load_best_model_at_end': True,
                'metric_for_best_model': 'eval_loss',
                'greater_is_better': False,
            }
        )
    else:
        training_kwargs.update(
            {
                eval_strategy_key: 'no',
                'save_strategy': 'steps',
                'load_best_model_at_end': False,
            }
        )

    return TrainingArguments(**training_kwargs)



def save_run_summary(path: Path, summary: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')



def main(argv: list[str]) -> int:
    args = parse_args(argv)
    ensure_exists(args.train_file, 'Training file')
    if args.validation_file and not args.validation_file.exists():
        print(f'Validation file not found, continuing without eval: {args.validation_file}')

    set_seed(args.seed)
    raw_datasets = load_text_datasets(args.train_file, args.validation_file)
    model, tokenizer = build_model_and_tokenizer(args)

    if not args.disable_gradient_checkpointing and hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()

    tokenized_datasets = tokenize_datasets(raw_datasets, tokenizer, max_length=args.max_length)
    has_validation = 'validation' in tokenized_datasets

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    training_args = build_training_args(args, has_validation=has_validation)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['validation'] if has_validation else None,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    args.output_dir.mkdir(parents=True, exist_ok=True)

    run_summary = {
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'model_name': args.model_name,
        'train_mode': args.train_mode,
        'train_file': str(args.train_file.resolve()),
        'validation_file': str(args.validation_file.resolve()) if args.validation_file.exists() else '',
        'output_dir': str(args.output_dir.resolve()),
        'parameter_counts': count_parameters(model),
        'max_length': args.max_length,
        'num_train_epochs': args.num_train_epochs,
        'learning_rate': choose_learning_rate(args),
        'per_device_train_batch_size': args.per_device_train_batch_size,
        'gradient_accumulation_steps': args.gradient_accumulation_steps,
        'fp16': torch.cuda.is_available(),
        'gradient_checkpointing': not args.disable_gradient_checkpointing,
        'has_validation': has_validation,
        'device': str(training_args.device),
    }
    save_run_summary(args.output_dir / 'run_config.json', run_summary)

    train_result = trainer.train(resume_from_checkpoint=args.resume_from_checkpoint)
    trainer.save_model()
    tokenizer.save_pretrained(args.output_dir)

    train_metrics = dict(train_result.metrics)
    trainer.log_metrics('train', train_metrics)
    trainer.save_metrics('train', train_metrics)
    trainer.save_state()

    if has_validation:
        eval_metrics = trainer.evaluate()
        trainer.log_metrics('eval', eval_metrics)
        trainer.save_metrics('eval', eval_metrics)

    return 0


if __name__ == '__main__':
    raise SystemExit(main(__import__('sys').argv[1:]))


In [ ]:
TRAIN_FILE = str(DATA_DIR / 'conditioned_train.jsonl')
VAL_FILE = str(DATA_DIR / 'conditioned_val.jsonl')
OUT_DIR = str(OUTPUT_DIR)

TRAIN_MODE = 'lora'
MAX_LENGTH = 256
EPOCHS = 5
TRAIN_BATCH = 1
EVAL_BATCH = 1
GRAD_ACCUM = 16

print('TRAIN_FILE =', TRAIN_FILE)
print('VAL_FILE =', VAL_FILE)
print('OUT_DIR =', OUT_DIR)
print('TRAIN_MODE =', TRAIN_MODE)


## Start Training

For a Colab T4, keep `TRAIN_MODE = 'lora'` unless you deliberately want to experiment with a much riskier full fine-tuning run.


In [ ]:
!python train_gpt2_large_hf.py \
  --model-name gpt2-large \
  --train-file "$TRAIN_FILE" \
  --validation-file "$VAL_FILE" \
  --output-dir "$OUT_DIR" \
  --train-mode "$TRAIN_MODE" \
  --max-length $MAX_LENGTH \
  --num-train-epochs $EPOCHS \
  --per-device-train-batch-size $TRAIN_BATCH \
  --per-device-eval-batch-size $EVAL_BATCH \
  --gradient-accumulation-steps $GRAD_ACCUM \
  --logging-steps 10 \
  --eval-steps 50 \
  --save-steps 50


## Quick Generation Test

This cell loads the trained adapter or model and generates a short Bhojpuri lyric sample.


In [ ]:
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_DIR = Path(OUT_DIR)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if (MODEL_DIR / 'adapter_config.json').exists():
    base_model = AutoModelForCausalLM.from_pretrained('gpt2-large')
    model = PeftModel.from_pretrained(base_model, MODEL_DIR)
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_DIR)

model.to(device)
model.eval()

prompt = 'Title: ??? ??????? ???
Artist: ??? ????
Language: Bhojpuri
Lyrics:
'
inputs = tokenizer(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        top_p=0.95,
        top_k=50,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))


## Optional: Zip The Output Folder

Run this if you want one archive to download from Colab.


In [ ]:
import shutil
archive_base = str(BASE_DIR / 'gpt2-large-bhojpuri')
zip_path = shutil.make_archive(archive_base, 'zip', root_dir=OUTPUT_DIR)
print('Created:', zip_path)
